# L0 · Modelos reales (Groq + Ollama `bge-m3`)

**Criterio de aceptación (docs/13, L0):** una llamada al LLM devuelve JSON válido y un lote de embeddings sale normalizado L2, con Groq + `bge-m3`.

Sin simulaciones: todas las llamadas van a los modelos reales configurados en `config/.env` (`LLM_PROVIDER`, `EMB_PROVIDER`).

### Solo en el entorno de la demo (Databricks)

En local estas dos celdas no hacen nada: la configuración sale de `config/.env`.

En Databricks:
1. La primera celda instala `requirements.txt` y reinicia Python.
2. En la segunda, **descomenta** las líneas: el token de Foundry se lee del secret scope con `dbutils` y el resto son parámetros no sensibles. Tienen que ejecutarse **antes** de importar `backend`.

In [1]:
import os

if "DATABRICKS_RUNTIME_VERSION" in os.environ:
    get_ipython().run_line_magic("pip", "install -q -r ../requirements.txt")
    dbutils.library.restartPython()  # noqa: F821 - dbutils existe solo en Databricks

In [2]:
# Descomentar en Databricks (después del reinicio de la celda anterior):
# import os
# os.environ["FOUNDRY_AI_TOKEN"] = dbutils.secrets.get(scope="<scope>", key="foundry_ai_token")
# os.environ["LLM_PROVIDER"] = "foundry"
# os.environ["EMB_PROVIDER"] = "foundry"   # en Databricks no hay Ollama
# os.environ["FOUNDRY_AI_ENDPOINT"] = "https://<recurso>.openai.azure.com"
# os.environ["FOUNDRY_AI_API_VERSION"] = "<versión>"
# os.environ["FOUNDRY_AI_DEPLOYMENT"] = "<deployment del LLM>"
# os.environ["FOUNDRY_AI_EMBED_DEPLOYMENT"] = "<deployment de embeddings>"

In [3]:
import sys, time, logging
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

import numpy as np
from pydantic import BaseModel, Field
from typing import Literal

from backend.config import settings as s
from backend.llm.client import ModelClient

{k: v for k, v in s.publico().items() if k.startswith(("LLM_", "EMB_", "GROQ_", "OLLAMA_"))}

{'LLM_PROVIDER': 'groq',
 'EMB_PROVIDER': 'ollama',
 'GROQ_API_KEY': '***',
 'GROQ_BASE_URL': 'https://api.groq.com/openai/v1',
 'GROQ_LLM_MODEL': 'openai/gpt-oss-120b',
 'OLLAMA_BASE_URL': 'http://localhost:11434/v1',
 'OLLAMA_LLM_MODEL': '',
 'OLLAMA_EMB_MODEL': 'bge-m3',
 'LLM_TEMPERATURE': 0.0,
 'LLM_TIMEOUT_S': 120.0,
 'EMB_BATCH': 64,
 'LLM_CONCURRENCY': 6,
 'LLM_RETRIES': 3}

## 1 · Salud: una llamada mínima real a cada modelo

In [4]:
cliente = ModelClient(s)
estado = cliente.ping()
estado

INFO httpx2: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


INFO httpx2: HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"


{'llm': {'ok': True, 'ms': 5776}, 'embeddings': {'ok': True, 'ms': 105}}

## 2 · LLM: JSON validado con Pydantic

Esquema y prompt de prueba, solo para este notebook (no son prompts del producto).

In [5]:
class Prueba(BaseModel):
    naturaleza: Literal["obligacion", "informativo"]
    sujeto_obligado: str | None
    obligacion: str | None
    confianza: float = Field(ge=0, le=1)

SISTEMA = (
    "Eres un analista normativo. Lee el texto y responde SOLO un objeto JSON con las claves "
    "naturaleza ('obligacion' o 'informativo'), sujeto_obligado (texto o null), "
    "obligacion (texto literal o null) y confianza (número entre 0 y 1)."
)
TEXTOS = {
    "obligacion": "Art. 46.- Los sujetos obligados deberán conservar, por un plazo mínimo de diez años, los registros de las operaciones y de la debida diligencia.",
    "informativo": "Art. 2.- Para efectos de esta ley, se entiende por beneficiario final a la persona natural que posee o controla en última instancia a un cliente.",
}

resultados, latencias = {}, []
for esperado, texto in TEXTOS.items():
    t0 = time.perf_counter()
    r = cliente.chat_json(SISTEMA, texto, Prueba)
    latencias.append(time.perf_counter() - t0)
    resultados[esperado] = r
    print(f"{esperado:12s} -> {r.naturaleza:12s} conf={r.confianza:.2f}  {latencias[-1]:.1f}s")
    print("   ", r.model_dump())

INFO httpx2: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


obligacion   -> obligacion   conf=0.99  0.8s
    {'naturaleza': 'obligacion', 'sujeto_obligado': 'Los sujetos obligados', 'obligacion': 'deberán conservar, por un plazo mínimo de diez años, los registros de las operaciones y de la debida diligencia.', 'confianza': 0.99}


INFO httpx2: HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


informativo  -> informativo  conf=0.99  0.7s
    {'naturaleza': 'informativo', 'sujeto_obligado': None, 'obligacion': None, 'confianza': 0.99}


## 3 · Embeddings: lote mayor que `EMB_BATCH` (prueba el troceo) y normalización L2

In [6]:
frases = [
    "Las entidades deberán conservar los registros por diez años.",
    "El área de cumplimiento archiva la documentación durante 10 años.",
    "El horario de atención al público es de 9 a 17 horas.",
]
lote = frases + [f"Texto de relleno número {i} para forzar más de un lote." for i in range(s.EMB_BATCH)]

t0 = time.perf_counter()
V = cliente.embed(lote)
t_emb = time.perf_counter() - t0
normas = np.linalg.norm(V, axis=1)
print(f"forma={V.shape} dtype={V.dtype} lotes={-(-len(lote) // s.EMB_BATCH)} tiempo={t_emb:.1f}s tasa={len(lote) / t_emb:.1f} textos/s")
print(f"norma min/max = {normas.min():.5f} / {normas.max():.5f}")
print(f"sim(norma, manual relacionado) = {V[0] @ V[1]:.3f}")
print(f"sim(norma, texto no relacionado) = {V[0] @ V[2]:.3f}")

INFO httpx2: HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"


INFO httpx2: HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"


forma=(67, 1024) dtype=float32 lotes=2 tiempo=9.7s tasa=6.9 textos/s
norma min/max = 1.00000 / 1.00000
sim(norma, manual relacionado) = 0.753
sim(norma, texto no relacionado) = 0.499


## 4 · Criterio de aceptación de L0

In [7]:
checks = {
    "ping LLM ok": estado["llm"]["ok"],
    "ping embeddings ok": estado["embeddings"]["ok"],
    "LLM devuelve JSON válido (Pydantic)": all(isinstance(r, Prueba) for r in resultados.values()),
    "LLM distingue obligación de informativo": resultados["obligacion"].naturaleza == "obligacion"
                                               and resultados["informativo"].naturaleza == "informativo",
    "embeddings normalizados L2": bool(np.allclose(normas, 1.0, atol=1e-4)),
    "embeddings: troceo en lotes conserva el orden": V.shape[0] == len(lote),
    "embeddings: relacionado > no relacionado": float(V[0] @ V[1]) > float(V[0] @ V[2]),
}
for k, v in checks.items():
    print(("✅" if v else "❌"), k)
print(f"\nLatencia LLM: min {min(latencias):.1f}s · max {max(latencias):.1f}s · media {np.mean(latencias):.1f}s · proveedor {s.LLM_PROVIDER}/{s.GROQ_LLM_MODEL if s.LLM_PROVIDER == 'groq' else ''}")
assert all(checks.values()), "L0 NO cumplido"
print("L0 cumplido")

✅ ping LLM ok
✅ ping embeddings ok
✅ LLM devuelve JSON válido (Pydantic)
✅ LLM distingue obligación de informativo
✅ embeddings normalizados L2
✅ embeddings: troceo en lotes conserva el orden
✅ embeddings: relacionado > no relacionado

Latencia LLM: min 0.7s · max 0.8s · media 0.8s · proveedor groq/openai/gpt-oss-120b
L0 cumplido
